# 02 - Préparation des Données

Ce notebook couvre la phase **Préparation des données** de CRISP-DM: imputation, encodage, standardisation, traitement du déséquilibre et split stratifié.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(ROOT / 'src'))

import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

from feature_engineering import load_streaming_dataset
from preprocessing import build_preprocessor, split_features_target, get_feature_types

## Chargement du Dataset Streaming

Le module de feature engineering applique le renommage métier et ajoute des variables comportementales simulées.

In [ ]:
df = load_streaming_dataset(ROOT / 'data' / 'raw' / 'IBM_Telco_Customer_Churn.csv')
X, y = split_features_target(df)
X.head()

## Identification des Types de Variables

Les variables numériques reçoivent imputation médiane et standardisation. Les variables catégorielles reçoivent imputation par modalité la plus fréquente et One-Hot Encoding.

In [ ]:
numeric_features, categorical_features = get_feature_types(X)
print('Numériques:', numeric_features)
print('Catégorielles:', categorical_features)

## Split Train/Test Stratifié 80/20

La stratification conserve la proportion de churners dans le train et le test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(y_train.mean(), y_test.mean())

## Pipeline de Preprocessing Réutilisable

Le pipeline évite la fuite de données: il apprend les transformations sur le train puis les applique au test.

In [ ]:
preprocessor = build_preprocessor(X_train)
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)
print(X_train_prepared.shape, X_test_prepared.shape)

## SMOTE sur les Données d’Entraînement Uniquement

SMOTE est appliqué après le split et uniquement au train. Le test reste inchangé afin de simuler de vrais nouveaux clients.

In [ ]:
smote = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = smote.fit_resample(X_train_prepared, y_train)
print('Avant SMOTE:', y_train.value_counts().to_dict())
print('Après SMOTE:', pd.Series(y_train_balanced).value_counts().to_dict())

## Export du Dataset Préparé

Le script d’entraînement exporte également le dataset adapté dans `data/processed/streaming_churn_processed.csv` pour audit et reproductibilité.

In [ ]:
processed_path = ROOT / 'data' / 'processed' / 'streaming_churn_processed.csv'
df.to_csv(processed_path, index=False)
processed_path

## Conclusion

Cette phase produit un pipeline réutilisable et compatible avec la validation croisée, l’entraînement et le scoring batch.